In [3]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from ipynb.fs.defs.competition1 import create_sparse_matrix, compute_stats
from surprise import SVD, Dataset, Reader, accuracy
from surprise.model_selection import cross_validate

In [4]:
file = "data/train.csv"
df = pd.read_csv(file, header=0)
y = df["rating"]
x = df.drop(columns="rating")
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.1, random_state=42, stratify=df.rating)

In [5]:
# ── Convertir tus DataFrames al formato de Surprise ──────────────────────────
# Surprise necesita un DataFrame con columnas [user, item, rating]
# y saber el rango de ratings

def prepare_surprise_data(x_train, y_train, x_val, y_val, rating_scale=(1, 10)):
    reader = Reader(rating_scale=rating_scale)

    train_df = x_train.copy()
    train_df.columns = ['user', 'item']
    train_df['rating'] = y_train.values

    val_df = x_val.copy()
    val_df.columns = ['user', 'item']
    val_df['rating'] = y_val.values

    # Surprise trabaja con su propio objeto Trainset
    train_data    = Dataset.load_from_df(train_df[['user', 'item', 'rating']], reader)
    trainset      = train_data.build_full_trainset()

    # Validación como lista de (uid, iid, rating)
    val_list = list(zip(val_df['user'], val_df['item'], val_df['rating']))

    return trainset, val_list


def train_surprise_pmf(x_train, y_train, x_val, y_val,
                       n_factors=20, lr=0.1, n_epochs=100,
                       reg_u=0.01, reg_i=0.01,
                       rating_scale=(1, 10),
                       user_means=None, item_means=None, global_mean=None,
                       cold_start_unknown_user='global_mean',
                       cold_start_unknown_item='user_mean',
                       cold_start_both='const_7'):
    """
    Entrena SVD de Surprise (equivalente a PMF cuando biases=False).
    Mismos hiperparámetros que tu clase PMF.

    Surprise SVD con biases=False es exactamente:
        r̂_ui = U[u] · V[i]
    que es la formulación estándar de PMF.
    """

    trainset, val_list = prepare_surprise_data(
        x_train, y_train, x_val, y_val, rating_scale
    )

    # biases=False → PMF puro (sin b_u ni b_i)
    # biases=True  → SVD completo (MF con sesgos, suele dar mejor resultado)
    model = SVD(
        n_factors  = n_factors,
        lr_all     = lr,
        reg_all    = reg_u,          # Surprise usa la misma reg para U y V
        n_epochs   = n_epochs,
        biased     = False,          # ← PMF puro, igual que tu clase
        random_state = 42
    )

    print("Training Surprise PMF (SVD biased=False)...")
    model.fit(trainset)

    # ── Evaluación en validación ──────────────────────────────────────────────
    def _cold(strat, uid=None, iid=None):
        if strat == 'item_mean' and iid is not None:
            # Buscar media del ítem en el trainset de Surprise
            try:
                iid_inner = trainset.to_inner_iid(str(iid))
                v = np.mean([r for (_, r) in trainset.ir[iid_inner]])
            except ValueError:
                v = np.nan
        elif strat == 'user_mean' and uid is not None:
            try:
                uid_inner = trainset.to_inner_uid(str(uid))
                v = np.mean([r for (_, r) in trainset.ur[uid_inner]])
            except ValueError:
                v = np.nan
        elif strat == 'global_mean':
            v = trainset.global_mean
        elif strat.startswith('const_'):
            v = float(strat.split('_')[1])
        else:
            v = np.nan
        return trainset.global_mean if np.isnan(v) else v

    preds_val, true_val = [], []
    for uid, iid, r_true in val_list:
        pred_obj   = model.predict(str(uid), str(iid))
        user_known = not pred_obj.details.get('was_impossible', False) or \
                     str(uid) in {trainset.to_raw_uid(i) for i in range(trainset.n_users)}
        item_known = str(iid) in {trainset.to_raw_iid(i) for i in range(trainset.n_items)}

        if pred_obj.details.get('was_impossible', False):
            # Cold start
            try:   trainset.to_inner_uid(str(uid)); u_known = True
            except ValueError: u_known = False
            try:   trainset.to_inner_iid(str(iid)); i_known = True
            except ValueError: i_known = False

            if not u_known and not i_known:
                pred = _cold(cold_start_both)
            elif not u_known:
                pred = _cold(cold_start_unknown_user, iid=iid)
            else:
                pred = _cold(cold_start_unknown_item, uid=uid)
        else:
            pred = np.clip(pred_obj.est, rating_scale[0], rating_scale[1])

        preds_val.append(pred)
        true_val.append(r_true)

    preds_val = np.array(preds_val)
    true_val  = np.array(true_val)
    mae  = np.mean(np.abs(preds_val - true_val))
    rmse = np.sqrt(np.mean((preds_val - true_val) ** 2))

    print(f"\n── Surprise PMF (SVD biased=False) ──")
    print(f"   Val MAE  : {mae:.4f}")
    print(f"   Val RMSE : {rmse:.4f}")

    return model, {'mae': mae, 'rmse': rmse}


def predict_surprise(model, x_test,
                     trainset,
                     user_means, item_means, global_mean,
                     cold_start_unknown_user='global_mean',
                     cold_start_unknown_item='user_mean',
                     cold_start_both='const_7',
                     clip_min=1.0, clip_max=10.0):
    """Genera predicciones para test con el mismo cold-start que tu PMF."""

    def _cold(strat, uid=None, iid=None):
        if strat == 'item_mean' and iid is not None:
            try:
                idx = trainset.to_inner_iid(str(iid))
                return np.mean([r for (_, r) in trainset.ir[idx]])
            except ValueError:
                return global_mean
        elif strat == 'user_mean' and uid is not None:
            try:
                idx = trainset.to_inner_uid(str(uid))
                return np.mean([r for (_, r) in trainset.ur[idx]])
            except ValueError:
                return global_mean
        elif strat == 'global_mean':
            return global_mean
        elif strat.startswith('const_'):
            return float(strat.split('_')[1])
        return global_mean

    ids, preds = [], []
    for _, row in tqdm(x_test.iterrows(), total=len(x_test), desc='Predicting'):
        row_id, user_id, item_id = row.iloc[0], row.iloc[1], row.iloc[2]
        ids.append(row_id)

        try:   trainset.to_inner_uid(str(user_id)); u_known = True
        except ValueError: u_known = False
        try:   trainset.to_inner_iid(str(item_id)); i_known = True
        except ValueError: i_known = False

        if not u_known and not i_known:
            preds.append(_cold(cold_start_both))
        elif not u_known:
            preds.append(_cold(cold_start_unknown_user, iid=item_id))
        elif not i_known:
            preds.append(_cold(cold_start_unknown_item, uid=user_id))
        else:
            est = model.predict(str(user_id), str(item_id)).est
            preds.append(np.clip(est, clip_min, clip_max))

    return pd.DataFrame({'id': ids, 'rating': preds})

In [ ]:
# Entrenar
surprise_model, surprise_metrics = train_surprise_pmf(
    x_train, y_train, x_val, y_val,
    n_factors  = 20,
    lr         = 0.1,
    n_epochs   = 100,
    reg_u      = 0.01,
    reg_i      = 0.01,
)


Training Surprise PMF (SVD biased=False)...

── Surprise PMF (SVD biased=False) ──
   Val MAE  : 1.5868
   Val RMSE : 1.9394


AttributeError: 'Trainset' object has no attribute 'build_full_trainset'

In [ ]:
# Predicciones test
trainset, _ = prepare_surprise_data(x_train, y_train, x_val, y_val)
trainset     = trainset.build_full_trainset()
'''
predictions = predict_surprise(
    surprise_model, x_test, trainset,
    user_means, item_means, global_mean,
    cold_start_unknown_user = 'global_mean',
    cold_start_unknown_item = 'user_mean',
    cold_start_both         = 'const_7'
)
'''